# Udemy Course Funnel & Segmentation Analysis

Goal: figure out which course categories/formats convert best from learner interest to completion, and where Udemy loses the most learners along the way.

## 1. Load the dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("emrebayirr/udemy-course-dataset-categories-ratings-and-trends")
path

Using Colab cache for faster access to the 'udemy-course-dataset-categories-ratings-and-trends' dataset.


'/kaggle/input/udemy-course-dataset-categories-ratings-and-trends'

In [ ]:
import os

os.listdir(path)

['udemy_courses.csv']

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(path, 'udemy_courses.csv'))
df.shape

(98104, 13)

In [ ]:
df.head()

,id,title,url,is_paid,instructor_names,category,headline,num_subscribers,rating,num_reviews,instructional_level,objectives,curriculum
0,567828,The Complete Python Bootcamp From Zero to Hero...,https://www.udemy.com/course/complete-python-b...,True,"Jose Portilla, Pierian Training",Development,Learn Python like a Professional Start from t...,1976866,4.576494,521219,All Levels,You will learn how to leverage the power of Py...,"Course Overview, Auto-Welcome Message, Course ..."
1,1565838,The Complete 2024 Web Development Bootcamp,https://www.udemy.com/course/the-complete-web-...,True,"Dr. Angela Yu, Developer and Lead Instructor",Development,Become a Full-Stack Web Developer with just ON...,1362586,4.679065,409793,All Levels,Build 16 web development projects for your por...,"Front-End Web Development, What You'll Get in ..."
2,2776760,100 Days of Code: The Complete Python Pro Boot...,https://www.udemy.com/course/100-days-of-code/,True,"Dr. Angela Yu, Developer and Lead Instructor",Development,Master Python by building 100 projects in 100 ...,1417942,4.698768,331803,All Levels,You will master the Python programming languag...,Day 1 - Beginner - Working with Variables in P...
3,625204,The Web Developer Bootcamp 2024,https://www.udemy.com/course/the-web-developer...,True,Colt Steele,Development,10 Hours of React just added. Become a Develop...,923815,4.673450,276723,All Levels,"The ins and outs of HTML5, CSS3, and Modern Ja...","Course Orientation, Welcome To The Course!, Jo..."
4,1362070,React - The Complete Guide 2024 (incl. Next.js...,https://www.udemy.com/course/react-the-complet...,True,"Academind by Maximilian Schwarzmüller, Maximil...",Development,Dive in and learn React.js from scratch! Learn...,909848,4.638643,220051,All Levels,Learn React from the ground up and finish the ...,"Getting Started, Welcome To The Course!, What ..."


In [ ]:
df.columns.tolist()

['id',
 'title',
 'url',
 'is_paid',
 'instructor_names',
 'category',
 'headline',
 'num_subscribers',
 'rating',
 'num_reviews',
 'instructional_level',
 'objectives',
 'curriculum']

## 2. Quick data quality check

Before doing anything else, checking for nulls, duplicates, and how the key columns are distributed.

In [ ]:
df.isnull().sum()

,0
id,0
title,0
url,0
is_paid,0
instructor_names,2
category,0
headline,0
num_subscribers,0
rating,0
num_reviews,0


In [ ]:
df.duplicated(subset='id').sum()

np.int64(0)

In [ ]:
df['category'].value_counts()

,count
category,
Development,9945
Business,9912
IT & Software,9888
Teaching & Academics,9763
Personal Development,9692
Design,9263
Health & Fitness,8559
Lifestyle,7426
Finance & Accounting,7324


In [ ]:
df['is_paid'].value_counts()

,count
is_paid,
True,93607
False,4497


In [ ]:
df['instructional_level'].value_counts()

,count
instructional_level,
All Levels,53354
Beginner Level,31551
Intermediate Level,11455
Expert Level,1744


In [ ]:
df[['num_subscribers', 'rating', 'num_reviews']].describe()

,num_subscribers,rating,num_reviews
count,9.810400e+04,98104.000000,98104.000000
mean,5.765752e+03,4.091346,533.024688
std,2.444085e+04,1.091701,4728.577300
min,0.000000e+00,0.000000,0.000000
25%,1.220000e+02,4.000639,12.000000
50%,7.540000e+02,4.389837,46.500000
75%,3.493250e+03,4.639675,165.000000
max,1.976866e+06,5.000000,521219.000000


Notes:
- No real nulls (2 missing instructor names, not important)
- No duplicate course ids
- `is_paid` is heavily imbalanced (93.6k paid vs 4.5k free)
- `num_subscribers` is very skewed (median 754, max ~2M) — a handful of huge courses will dominate raw totals, so use medians/rates not sums when comparing groups
- `rating` has some 0s, likely courses with no ratings yet

## 3. Building a funnel

The dataset doesn't have actual funnel/traffic data, so this is simulated — but anchored to real columns so it's not just random numbers. `num_subscribers` is treated as ground truth (`enrolled`), and the earlier stages are derived backwards from it. `completed` is estimated using review engagement as a proxy for how many people stuck with the course.

In [ ]:
# enrolled = actual subscriber count, this is real data
df['enrolled'] = df['num_subscribers']

In [ ]:
import numpy as np
np.random.seed(42)

# sign_ups -> enrolled: paid courses have more drop-off here (you have to actually pay)
# free courses convert sign-up to enrollment almost 1:1
signup_to_enroll_rate = np.where(
    df['is_paid'],
    np.random.uniform(0.5, 0.75, len(df)),
    np.random.uniform(0.85, 0.98, len(df))
)

df['sign_ups'] = np.ceil(df['enrolled'] / signup_to_enroll_rate).astype(int)

In [ ]:
# page_visits -> sign_ups: higher-rated courses convert visitors to sign-ups better
rating_filled = df['rating'].replace(0, np.nan)
rating_filled = rating_filled.fillna(df.groupby('category')['rating'].transform('median'))
rating_signal = (rating_filled / 5).clip(0.3, 1)

visit_to_signup_rate = np.random.uniform(0.15, 0.25, len(df)) * rating_signal
df['page_visits'] = np.ceil(df['sign_ups'] / visit_to_signup_rate).astype(int)

In [ ]:
# completed: proxied using review engagement (more reviews per subscriber ~ more engaged learners)
# plus a small boost for higher course levels (advanced learners tend to finish what they start)
engagement_ratio = (df['num_reviews'] / df['num_subscribers'].replace(0, np.nan)).fillna(0).clip(0, 1)

level_completion_boost = df['instructional_level'].map({
    'Beginner Level': 0.9,
    'All Levels': 1.0,
    'Intermediate Level': 1.1,
    'Expert Level': 1.25
})

base_completion_rate = np.random.uniform(0.2, 0.4, len(df))
df['completed'] = (df['enrolled'] * base_completion_rate * (1 + engagement_ratio) * level_completion_boost).round().astype(int)
df['completed'] = df['completed'].clip(upper=df['enrolled'])

## 4. Sanity check

Making sure the funnel actually makes sense: visits >= sign-ups >= enrolled >= completed, no broken values.

In [ ]:
(df['page_visits'] >= df['sign_ups']).all(), \
(df['sign_ups'] >= df['enrolled']).all(), \
(df['enrolled'] >= df['completed']).all()

(np.True_, np.True_, np.True_)

In [ ]:
df[['page_visits','sign_ups','enrolled','completed']].describe()

,page_visits,sign_ups,enrolled,completed
count,9.810400e+04,9.810400e+04,9.810400e+04,98104.000000
mean,5.426015e+04,9.184119e+03,5.765752e+03,1844.594716
std,2.318605e+05,3.994794e+04,2.444085e+04,8528.681053
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,1.144000e+03,1.950000e+02,1.220000e+02,41.000000
50%,6.959500e+03,1.183000e+03,7.540000e+02,239.000000
75%,3.239325e+04,5.503000e+03,3.493250e+03,1102.000000
max,1.752670e+07,3.330104e+06,1.976866e+06,941320.000000


In [ ]:
df[['category','is_paid','num_subscribers','page_visits','sign_ups','enrolled','completed']].head(10)

,category,is_paid,num_subscribers,page_visits,sign_ups,enrolled,completed
0,Development,True,1976866,17526699,3330104,1976866,941320
1,Development,True,1362586,12176024,1847127,1362586,414985
2,Development,True,1417942,11624281,2076055,1417942,580398
3,Development,True,923815,7410942,1421988,923815,387452
4,Development,True,909848,10591754,1688016,909848,283384
5,Development,True,948984,7867047,1760643,948984,332713
6,Development,True,788455,7888168,1532407,788455,201707
7,Development,True,884140,8308309,1233895,884140,297817
8,Development,True,1093550,11710479,1681664,1093550,361138
9,Development,True,737929,6043450,1089970,737929,195802


## 5. Conversion rates at each stage

In [ ]:
df['visit_to_signup_rate'] = df['sign_ups'] / df['page_visits']
df['signup_to_enroll_rate'] = df['enrolled'] / df['sign_ups']
df['enroll_to_complete_rate'] = df['completed'] / df['enrolled']
df['overall_conversion_rate'] = df['completed'] / df['page_visits']

### By category

In [ ]:
df.groupby('category')[['visit_to_signup_rate','signup_to_enroll_rate','enroll_to_complete_rate']] \
    .mean().sort_values('enroll_to_complete_rate')

,visit_to_signup_rate,signup_to_enroll_rate,enroll_to_complete_rate
category,,,
Marketing,0.169088,0.659814,0.318452
Music,0.178753,0.637886,0.320928
Photography & Video,0.172315,0.640576,0.321573
Design,0.171644,0.643216,0.322918
Health & Fitness,0.179140,0.644271,0.326535
Development,0.170142,0.623386,0.327748
Lifestyle,0.178143,0.636649,0.329326
Finance & Accounting,0.169635,0.649590,0.331142
IT & Software,0.171080,0.623362,0.337950


### Paid vs free

In [ ]:
df.groupby('is_paid')[['visit_to_signup_rate','signup_to_enroll_rate','enroll_to_complete_rate']].mean()

,visit_to_signup_rate,signup_to_enroll_rate,enroll_to_complete_rate
is_paid,,,
False,0.173984,0.914766,0.293592
True,0.172960,0.619732,0.333968


### By course level

In [ ]:
df.groupby('instructional_level')[['visit_to_signup_rate','signup_to_enroll_rate','enroll_to_complete_rate']].mean()

,visit_to_signup_rate,signup_to_enroll_rate,enroll_to_complete_rate
instructional_level,,,
All Levels,0.172776,0.630126,0.337722
Beginner Level,0.173396,0.641378,0.302649
Expert Level,0.172057,0.625974,0.423451
Intermediate Level,0.173170,0.628947,0.372305


Paid courses convert worse at sign-up -> enroll (real payment friction) but finish better once people are in (33% vs 29% for free). That's a genuine, usable finding.

The course-level gap (Beginner 30% vs Expert 42% completion) is expected — it's built into how `completed` was modeled, not a discovery. Worth being upfront about that in the writeup rather than presenting it as a finding.

## 6. Segmenting courses

Same idea as the RFM segmentation from the Chinook project, applied here using completion rate and popularity instead of recency/frequency/monetary.

In [ ]:
# a few courses have 0 subscribers -> 0 enrolled -> can't compute a completion rate for them
(df['enrolled'] == 0).sum()

np.int64(1799)

In [ ]:
# set those aside rather than let them break the quartile scoring
df_seg = df[df['enrolled'] > 0].copy()
df_seg['enroll_to_complete_rate'] = df_seg['completed'] / df_seg['enrolled']

In [ ]:
df_seg['completion_score'] = pd.qcut(df_seg['enroll_to_complete_rate'], 4, labels=[1,2,3,4], duplicates='drop')
df_seg['popularity_score'] = pd.qcut(df_seg['num_subscribers'].rank(method='first'), 4, labels=[1,2,3,4])

In [ ]:
def segment_course(row):
    c, p = int(row['completion_score']), int(row['popularity_score'])
    if c >= 3 and p >= 3:
        return 'Star Performer'          # high completion, high popularity
    elif c >= 3 and p < 3:
        return 'Hidden Gem'              # high completion, low popularity - undermarketed
    elif c < 3 and p >= 3:
        return 'High-Interest Drop-off'  # popular but low completion - content/pacing issue
    else:
        return 'Underperformer'

df_seg['course_segment'] = df_seg.apply(segment_course, axis=1)
df_seg['course_segment'].value_counts()

,count
course_segment,
High-Interest Drop-off,27213
Hidden Gem,27213
Underperformer,20940
Star Performer,20939


In [ ]:
# bring the segment back into the main dataframe, courses with 0 enrolled get their own label
df['course_segment'] = df_seg['course_segment']
df['course_segment'] = df['course_segment'].fillna('No Activity')
df['course_segment'].value_counts()

,count
course_segment,
High-Interest Drop-off,27213
Hidden Gem,27213
Underperformer,20940
Star Performer,20939
No Activity,1799


## 7. What's actually in each segment

**Biggest "High-Interest Drop-off" courses** — popular courses losing the most learners:

In [ ]:
df[df['course_segment'] == 'High-Interest Drop-off'] \
    .nlargest(5, 'num_subscribers')[['title','category','num_subscribers','enroll_to_complete_rate']]

,title,category,num_subscribers,enroll_to_complete_rate
1,The Complete 2024 Web Development Bootcamp,Development,1362586,0.304557
27182,[NEW] Ultimate AWS Certified Cloud Practitione...,IT & Software,1049095,0.312319
27181,Ultimate AWS Certified Solutions Architect Ass...,IT & Software,1039290,0.252478
4,React - The Complete Guide 2024 (incl. Next.js...,Development,909848,0.311463
59979,The Complete Digital Marketing Course - 12 Cou...,Marketing,803408,0.301369


**"Hidden Gem" courses** — need a minimum subscriber count here, otherwise a course with 1-2 subscribers and a lucky completion looks like a perfect 100% "gem" which isn't a real signal:

In [ ]:
hidden_gems = df[(df['course_segment'] == 'Hidden Gem') & (df['num_subscribers'] >= 100)]
hidden_gems.nlargest(5, 'enroll_to_complete_rate')[['title','category','num_subscribers','enroll_to_complete_rate']]

,title,category,num_subscribers,enroll_to_complete_rate
38356,Product Management Essentials: What you need t...,Office Productivity,112,0.687500
13232,Social Recruitment Fast Track,Business,529,0.676749
13516,"Part 2/4-Document Control-Workflow Management,...",Business,440,0.661364
38651,Power Pivot Workshop Advanced,Office Productivity,168,0.654762
16855,Policy and Procedure Best Practices: Managemen...,Business,120,0.650000


**Which categories show up most in each segment:**

In [ ]:
df.groupby('course_segment')['category'].value_counts(normalize=True).groupby(level=0).head(3)

course_segment          category            
Hidden Gem              Teaching & Academics    0.126337
                        Personal Development    0.124389
                        Health & Fitness        0.114614
High-Interest Drop-off  Development             0.163304
                        IT & Software           0.132290
                        Business                0.121045
No Activity             Health & Fitness        0.225681
                        Finance & Accounting    0.171762
                        Marketing               0.148416
Star Performer          Development             0.174077
                        IT & Software           0.158890
                        Business                0.142079
Underperformer          Health & Fitness        0.125597
                        Teaching & Academics    0.117765
                        Design                  0.115664
Name: proportion, dtype: float64

Takeaways:
- Drop-off is concentrated in Development and IT & Software — ironically Udemy's biggest, most popular courses (Web Dev Bootcamp, AWS certs, React). High enrollment but a lot of people don't finish, probably a course length/pacing issue rather than quality.
- Hidden Gems cluster around niche Business/Office Productivity topics (product management, document control, Power Pivot) — solid completion rates, just under-discovered.
- Paid courses retain better than free ones once someone's actually enrolled.

In [ ]:
# columns that match the MySQL table structure
cols = ['id','title','is_paid','category','num_subscribers','rating','num_reviews',
        'instructional_level','enrolled','sign_ups','page_visits','completed',
        'visit_to_signup_rate','signup_to_enroll_rate','enroll_to_complete_rate',
        'overall_conversion_rate','course_segment']

df_sql = df[cols].copy()

# fix is_paid: True/False -> 1/0 (MySQL BOOLEAN column needs this)
df_sql['is_paid'] = df_sql['is_paid'].astype(int)

# round decimals to fit the column definitions (this is what caused the import errors)
df_sql['rating'] = df_sql['rating'].round(3)          # fits DECIMAL(4,3)
for c in ['visit_to_signup_rate','signup_to_enroll_rate',
          'enroll_to_complete_rate','overall_conversion_rate']:
    df_sql[c] = df_sql[c].round(6)                      # fits DECIMAL(10,6)

# strip double quotes from titles (breaks MySQL's CSV parser otherwise)
df_sql['title'] = df_sql['title'].str.replace('"', '', regex=False)

# save and download
df_sql.to_csv('courses_final.csv', index=False)
print(df_sql.shape)

(98104, 17)
